# 03. Бюджет контекста

Контекстное окно не бесконечно. Даже если модель поддерживает длинный контекст, агенту все равно нужен бюджет: что кладем в запрос, что режем, что саммарим, а что достаем через retrieval позже.

Ниже не настоящий tokenizer, а учебная оценка «примерных токенов». Для первого прохода этого достаточно, чтобы увидеть саму механику отбора.

In [ ]:
candidates = [
    {"name": "developer_rules", "tokens": 90, "relevance": 10, "trust": 10},
    {"name": "current_task", "tokens": 35, "relevance": 10, "trust": 8},
    {"name": "run_state", "tokens": 55, "relevance": 8, "trust": 9},
    {"name": "wiki_page_aurora_17", "tokens": 240, "relevance": 9, "trust": 7},
    {"name": "source_postmortem_excerpt", "tokens": 210, "relevance": 9, "trust": 4},
    {"name": "old_customer_chat", "tokens": 180, "relevance": 4, "trust": 5},
    {"name": "full_raw_log", "tokens": 600, "relevance": 3, "trust": 6},
]

budget = 520

def priority(item):
    return (item["relevance"] * 2 + item["trust"]) / item["tokens"]


selected = []
dropped = []
used = 0

for item in sorted(candidates, key=priority, reverse=True):
    if used + item["tokens"] <= budget:
        selected.append(item)
        used += item["tokens"]
    else:
        dropped.append(item)

print(f"Бюджет: {budget}")
print(f"Использовано: {used}")
print("\nВ контекст пошло:")
for item in selected:
    print(f"- {item['name']} ({item['tokens']} токенов)")

print("\nНе влезло:")
for item in dropped:
    print(f"- {item['name']} ({item['tokens']} токенов)")

Обратите внимание: в реальном агенте мы бы не просто выбрасывали `full_raw_log`. Мы могли бы саммарить его, положить ссылку на артефакт, оставить только релевантный фрагмент или дать агенту инструмент для чтения лога по запросу.

In [ ]:
def compaction_hint(dropped_items):
    hints = []
    for item in dropped_items:
        if item["relevance"] >= 7:
            hints.append(f"Сжать {item['name']} до 80-120 токенов и попробовать снова.")
        else:
            hints.append(f"Оставить {item['name']} вне контекста, читать только через инструмент.")
    return hints


for hint in compaction_hint(dropped):
    print("-", hint)